In [1]:
import threading
import time

import pandas as pd
from pylsl import StreamInfo, StreamOutlet, StreamInlet, resolve_byprop

In [2]:
# Configuration
FILE_PATH = "../../data/all_trials_25_hz_stacked_null_str_filled.csv"
SAMPLING_RATE = 25  # 25 Hz
INTERVAL = 1.0 / SAMPLING_RATE  # 0.04 seconds per sample

In [3]:
class EquivitalLSLStreamer:
    def __init__(self, csv_filepath, srate=25.0, stream_name="Equivital_Stream"):
        self.csv_filepath = csv_filepath
        self.srate = srate
        self.interval = 1.0 / srate
        self.stream_name = stream_name

        self.df = None
        self.columns = []
        self.data = []
        self.num_channels = 0

        # Internal threading control
        self.is_running = False
        self._thread = None

        # Load and prepare data (now memory-optimized)
        self._load_and_prepare_data()
        # Initialize the LSL outlet
        self._setup_lsl()

    def _load_and_prepare_data(self):
        print(f"Analyzing CSV headers in {self.csv_filepath}...")

        # 1. Read ONLY the header row (nrows=0) to prevent OOM
        header_df = pd.read_csv(self.csv_filepath, nrows=0)

        # Filter out which columns contain "equivital" (case-insensitive)
        equivital_cols = [col for col in header_df.columns if "equivital" in col.lower()]

        if not equivital_cols:
            raise ValueError("No columns containing the word 'Equivital' were found in the CSV headers.")

        print(f"Found {len(equivital_cols)} Equivital columns. Loading only these columns into memory...")

        # 2. Read ONLY the specified columns from the CSV using 'usecols'
        # This keeps memory usage minimal as irrelevant columns are completely ignored during parsing.
        self.df = pd.read_csv(self.csv_filepath, usecols=equivital_cols)

        # Ensure columns are ordered the way pandas loaded them
        self.columns = self.df.columns.tolist()

        # Fill missing values: forward-fill first, then replace any remaining NaNs with 0.0
        self.df = self.df.ffill().fillna(0.0)

        # Convert to list for efficient iteration during streaming
        self.data = self.df.values.tolist()
        self.num_channels = len(self.columns)

        print(f"Successfully loaded {len(self.data)} rows for {self.num_channels} columns.")

    def _setup_lsl(self):
        # Configure LSL metadata
        self.info = StreamInfo(
            name=self.stream_name,
            type="Physiological",
            channel_count=self.num_channels,
            nominal_srate=self.srate,
            channel_format="float32",
            source_id="equivital_sim_25hz"
        )

        # Append channel labels so receiver programs know which column is which
        desc = self.info.desc()
        channels = desc.append_child("channels")
        for col in self.columns:
            ch = channels.append_child("channel")
            ch.append_child_value("label", col)

        self.outlet = StreamOutlet(self.info)
        print("LSL Outlet initialized successfully.")

    def _stream_loop(self):
        n_samples = len(self.data)
        idx = 0

        # Record the start time to calculate intervals dynamically
        start_time = time.perf_counter()

        while self.is_running:
            # Wrap around to the beginning if we reach the end of the CSV data
            sample = self.data[idx % n_samples]

            # Push the sample to LSL
            self.outlet.push_sample(sample)

            idx += 1

            # Calculate next target time to maintain a stable 25Hz rate,
            # which minimizes timing drift compared to simple sleep intervals
            target_time = start_time + (idx * self.interval)
            sleep_duration = target_time - time.perf_counter()

            if sleep_duration > 0:
                time.sleep(sleep_duration)

    def start(self):
        """Starts streaming in a background thread."""
        if not self.is_running:
            self.is_running = True
            # daemon=True ensures the thread exits when the Jupyter kernel stops or restarts
            self._thread = threading.Thread(target=self._stream_loop, daemon=True)
            self._thread.start()
            print(f"Background thread started. Streaming at {self.srate} Hz...")
        else:
            print("Streamer is already running.")

    def stop(self):
        """Stops the background streaming thread."""
        if self.is_running:
            self.is_running = False
            if self._thread:
                self._thread.join(timeout=1.0)
            print("Streamer successfully stopped.")
        else:
            print("Streamer is not currently running.")

In [4]:
streamer = EquivitalLSLStreamer(csv_filepath=FILE_PATH, srate=SAMPLING_RATE)

Analyzing CSV headers in ../../data/all_trials_25_hz_stacked_null_str_filled.csv...
Found 13 Equivital columns. Loading only these columns into memory...
Successfully loaded 1702139 rows for 13 columns.
LSL Outlet initialized successfully.


2026-07-04 22:27:24.158 (  57.152s) [        459D9740]         api_config.cpp:126   INFO| Loaded default config
2026-07-04 22:27:24.170 (  57.164s) [        459D9740]             common.cpp:78    INFO| git:64988c6a14b8dc3b3f270ece58eab4f480bfab43/branch:refs/tags/v1.17.7/build:Release/compiler:GNU-11.4.0/link:SHARED


In [9]:
streamer.df.isnull().sum()

HR (bpm) - Equivital                                  0
BR (rpm) - Equivital                                  0
Skin Temperature - IR Thermometer (°C) - Equivital    0
ECG Lead 1 - Equivital                                0
ECG Lead 2 - Equivital                                0
Lateral Acc - Equivital                               0
Longitudinal Acc - Equivital                          0
Vertical Acc - Equivital                              0
magnitude - Equivital                                 0
Time (s) - Equivital                                  0
HR_instant - Equivital                                0
HR_average - Equivital                                0
HR_w_average - Equivital                              0
dtype: int64

In [12]:
streamer.df.iloc[:, 2].unique()

array([34.3    , 34.30046, 34.30126, ..., 35.09616, 35.09776, 35.09936],
      shape=(118125,))

In [5]:
streamer.start()

Background thread started. Streaming at 25 Hz...


In [6]:

# 1. Resolve the stream by its 'name' property
print("Searching for the 'Equivital_Stream' on the network...")
streams = resolve_byprop('name', 'Equivital_Stream', timeout=5.0)

if not streams:
    print("No stream named 'Equivital_Stream' was found. Is your simulator running?")
else:
    # 2. Connect to the stream
    inlet = StreamInlet(streams[0])
    print(f"Successfully connected to: {streams[0].name()}")

    # 3. Retrieve channel names from stream metadata
    info = inlet.info()
    desc = info.desc()
    channels = desc.child("channels")
    labels = []
    if not channels.empty():
        ch = channels.child("channel")
        while not ch.empty():
            labels.append(ch.child_value("label"))
            ch = ch.next_sibling("channel")
    print(f"Channels: {labels}")

    # 4. Pull and print 10 samples to verify
    print("Reading data...")
    for _ in range(10):
        sample, timestamp = inlet.pull_sample()
        print(f"TS: {timestamp:.3f} | Data: {sample}")

Searching for the 'Equivital_Stream' on the network...
Successfully connected to: Equivital_Stream
Channels: ['HR (bpm) - Equivital', 'BR (rpm) - Equivital', 'Skin Temperature - IR Thermometer (°C) - Equivital', 'ECG Lead 1 - Equivital', 'ECG Lead 2 - Equivital', 'Lateral Acc - Equivital', 'Longitudinal Acc - Equivital', 'Vertical Acc - Equivital', 'magnitude - Equivital', 'Time (s) - Equivital', 'HR_instant - Equivital', 'HR_average - Equivital', 'HR_w_average - Equivital']
Reading data...
TS: 2018.326 | Data: [84.63922119140625, 12.699999809265137, 34.29999923706055, -0.11999999731779099, -0.10999999940395355, -325.8999938964844, 415.1000061035156, -1063.5999755859375, 1187.344970703125, 2450.988037109375, 79.29325866699219, 82.8980712890625, 83.5634536743164]
TS: 2018.366 | Data: [84.58081817626953, 12.699999809265137, 34.29999923706055, -0.10999999940395355, -0.10000000149011612, -332.70001220703125, 414.29998779296875, -1062.0, 1187.5245361328125, 2451.028076171875, 79.29325866699

In [7]:
streamer.stop()

Streamer successfully stopped.
